# Tugas 4 No. 2 — Pembobotan Term (TF-IDF)
**Topik:** Bag-of-Words, Term Frequency (TF), Inverse Document Frequency (IDF), dan TF-IDF

**Studi Kasus:** Prototipe sistem pencarian dokumen teknis untuk mahasiswa Teknik Komputer.
Dokumen-dokumen singkat diubah menjadi representasi numerik (TF-IDF) agar nantinya dapat
digunakan untuk perhitungan kemiripan (*cosine similarity*).

**Nama :** _Zahrah Aliyah Zairah_
**NIM  :** _240210501068_


## 0. Import Library


In [9]:
#install sastrawi
!pip install Sastrawi -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.3 MB/s eta 0:00:00


In [10]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Sastrawi bersifat opsional pada tugas ini. Jika tersedia, gunakan daftar stopword-nya;
# jika tidak tersedia di environment, gunakan daftar stopword cadangan (fallback) di bawah.
try:
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    daftar_stopword = set(StopWordRemoverFactory().get_stop_words())
    print("Menggunakan daftar stopword dari Sastrawi, jumlah:", len(daftar_stopword))
except ImportError:
    daftar_stopword = {
        "yang", "dan", "di", "ke", "dari", "untuk", "pada", "dengan", "akan",
        "telah", "adalah", "ini", "itu", "atau", "juga", "dalam", "secara",
        "dapat", "agar", "para", "oleh", "sebagai", "tersebut", "bagi",
    }
    print("Sastrawi tidak ditemukan, menggunakan daftar stopword cadangan, jumlah:",
          len(daftar_stopword))

Menggunakan daftar stopword dari Sastrawi, jumlah: 123


## 1. Dataset


In [11]:
dokumen = [
    "Sistem komputer modern menggunakan prosesor yang sangat cepat untuk mengolah data.",
    "Jaringan komputer menghubungkan banyak perangkat agar dapat saling bertukar data.",
    "Kecerdasan buatan adalah cabang ilmu komputer yang mempelajari cara membuat mesin cerdas.",
    "Sistem temu kembali informasi membantu pengguna menemukan dokumen yang relevan dengan cepat.",
]

for i, d in enumerate(dokumen, 1):
    print(f"Dokumen {i}: {d}")

Dokumen 1: Sistem komputer modern menggunakan prosesor yang sangat cepat untuk mengolah data.
Dokumen 2: Jaringan komputer menghubungkan banyak perangkat agar dapat saling bertukar data.
Dokumen 3: Kecerdasan buatan adalah cabang ilmu komputer yang mempelajari cara membuat mesin cerdas.
Dokumen 4: Sistem temu kembali informasi membantu pengguna menemukan dokumen yang relevan dengan cepat.


## 2. Preprocessing Sederhana


In [12]:
def simple_preprocess(text):
    # Case folding
    text = text.lower()
    # Cleaning ringan: hanya sisakan huruf & spasi
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    # Tokenisasi
    tokens = text.split()
    # Stopwords removal
    tokens = [t for t in tokens if t not in daftar_stopword]
    return tokens

tokenized_docs = [simple_preprocess(d) for d in dokumen]
processed_docs = [" ".join(tok) for tok in tokenized_docs]  # untuk input TfidfVectorizer

for i, tok in enumerate(tokenized_docs, 1):
    print(f"Dokumen {i} (setelah preprocessing): {tok}")

Dokumen 1 (setelah preprocessing): ['sistem', 'komputer', 'modern', 'menggunakan', 'prosesor', 'sangat', 'cepat', 'mengolah', 'data']
Dokumen 2 (setelah preprocessing): ['jaringan', 'komputer', 'menghubungkan', 'banyak', 'perangkat', 'saling', 'bertukar', 'data']
Dokumen 3 (setelah preprocessing): ['kecerdasan', 'buatan', 'cabang', 'ilmu', 'komputer', 'mempelajari', 'cara', 'membuat', 'mesin', 'cerdas']
Dokumen 4 (setelah preprocessing): ['sistem', 'temu', 'informasi', 'membantu', 'pengguna', 'menemukan', 'dokumen', 'relevan', 'cepat']


## 3. Bag-of-Words (Raw Count)

In [13]:
vocab = sorted(set(w for doc in tokenized_docs for w in doc))
print("Ukuran vocabulary:", len(vocab))
print(vocab)

N = len(dokumen)  # jumlah dokumen

bow_data = {term: [doc.count(term) for doc in tokenized_docs] for term in vocab}
df_bow = pd.DataFrame(bow_data, index=[f"Doc{i+1}" for i in range(N)]).T
df_bow.index.name = "Term"
df_bow

Ukuran vocabulary: 31
['banyak', 'bertukar', 'buatan', 'cabang', 'cara', 'cepat', 'cerdas', 'data', 'dokumen', 'ilmu', 'informasi', 'jaringan', 'kecerdasan', 'komputer', 'membantu', 'membuat', 'mempelajari', 'menemukan', 'menggunakan', 'menghubungkan', 'mengolah', 'mesin', 'modern', 'pengguna', 'perangkat', 'prosesor', 'relevan', 'saling', 'sangat', 'sistem', 'temu']


,Doc1,Doc2,Doc3,Doc4
Term,,,,
banyak,0,1,0,0
bertukar,0,1,0,0
buatan,0,0,1,0
cabang,0,0,1,0
cara,0,0,1,0
cepat,1,0,0,1
cerdas,0,0,1,0
data,1,1,0,0
dokumen,0,0,0,1


## 4. Perhitungan Manual: TF, DF, IDF, dan TF-IDF


In [14]:
# --- Term Frequency (TF) = raw count (sudah dihitung sebagai df_bow) ---
df_tf = df_bow.copy()
print("=== Term Frequency (TF) ===")
display(df_tf)

# --- Document Frequency (DF) ---
df_document_freq = (df_bow > 0).sum(axis=1)
print("\n=== Document Frequency (DF) ===")
display(df_document_freq.to_frame(name="DF"))

# --- Inverse Document Frequency (IDF), rumus smoothed (selaras dengan scikit-learn) ---
idf_manual = np.log((1 + N) / (1 + df_document_freq)) + 1
print("\n=== Inverse Document Frequency (IDF) ===")
display(idf_manual.to_frame(name="IDF"))

=== Term Frequency (TF) ===


,Doc1,Doc2,Doc3,Doc4
Term,,,,
banyak,0,1,0,0
bertukar,0,1,0,0
buatan,0,0,1,0
cabang,0,0,1,0
cara,0,0,1,0
cepat,1,0,0,1
cerdas,0,0,1,0
data,1,1,0,0
dokumen,0,0,0,1



=== Document Frequency (DF) ===


,DF
Term,
banyak,1
bertukar,1
buatan,1
cabang,1
cara,1
cepat,2
cerdas,1
data,2
dokumen,1



=== Inverse Document Frequency (IDF) ===


,IDF
Term,
banyak,1.916291
bertukar,1.916291
buatan,1.916291
cabang,1.916291
cara,1.916291
cepat,1.510826
cerdas,1.916291
data,1.510826
dokumen,1.916291


In [15]:
# --- TF-IDF manual = TF x IDF, lalu normalisasi L2 per dokumen ---
tfidf_manual = df_tf.mul(idf_manual, axis=0)

norm_l2 = np.sqrt((tfidf_manual ** 2).sum(axis=0))
tfidf_manual_norm = tfidf_manual.div(norm_l2, axis=1)

print("=== Matriks TF-IDF (Perhitungan Manual, sudah dinormalisasi L2) ===")
tfidf_manual_norm.round(4)

=== Matriks TF-IDF (Perhitungan Manual, sudah dinormalisasi L2) ===


,Doc1,Doc2,Doc3,Doc4
Term,,,,
banyak,0.0000,0.3772,0.0000,0.0000
bertukar,0.0000,0.3772,0.0000,0.0000
buatan,0.0000,0.0000,0.3260,0.0000
cabang,0.0000,0.0000,0.3260,0.0000
cara,0.0000,0.0000,0.3260,0.0000
cepat,0.2924,0.0000,0.0000,0.2746
cerdas,0.0000,0.0000,0.3260,0.0000
data,0.2924,0.2974,0.0000,0.0000
dokumen,0.0000,0.0000,0.0000,0.3483


## 5. Perhitungan TF-IDF dengan `scikit-learn` (`TfidfVectorizer`)

In [16]:
vectorizer = TfidfVectorizer(vocabulary=vocab)  # vocabulary disamakan agar mudah dibandingkan
X = vectorizer.fit_transform(processed_docs)

tfidf_sklearn = pd.DataFrame(
    X.toarray().T,
    index=vectorizer.get_feature_names_out(),
    columns=[f"Doc{i+1}" for i in range(N)],
)
tfidf_sklearn.index.name = "Term"

print("=== Matriks TF-IDF (scikit-learn) ===")
tfidf_sklearn.round(4)

=== Matriks TF-IDF (scikit-learn) ===


,Doc1,Doc2,Doc3,Doc4
Term,,,,
banyak,0.0000,0.3772,0.0000,0.0000
bertukar,0.0000,0.3772,0.0000,0.0000
buatan,0.0000,0.0000,0.3260,0.0000
cabang,0.0000,0.0000,0.3260,0.0000
cara,0.0000,0.0000,0.3260,0.0000
cepat,0.2924,0.0000,0.0000,0.2746
cerdas,0.0000,0.0000,0.3260,0.0000
data,0.2924,0.2974,0.0000,0.0000
dokumen,0.0000,0.0000,0.0000,0.3483


## 6. Perbandingan Hasil Manual vs scikit-learn

In [17]:
selisih = (tfidf_manual_norm.loc[vocab] - tfidf_sklearn.loc[vocab]).abs()
print("Selisih absolut maksimum antara perhitungan manual dan scikit-learn:")
print(selisih.values.max())

print("\nKesimpulan: perhitungan manual dan hasil scikit-learn menghasilkan nilai yang "
      "(hampir) identik, dengan selisih hanya pada tingkat pembulatan angka floating-point.")

Selisih absolut maksimum antara perhitungan manual dan scikit-learn:
5.551115123125783e-17

Kesimpulan: perhitungan manual dan hasil scikit-learn menghasilkan nilai yang (hampir) identik, dengan selisih hanya pada tingkat pembulatan angka floating-point.


## 7. Analisis Singkat

Pada Dokumen 1, term dengan bobot TF-IDF tertinggi adalah kata-kata yang hanya muncul di
dokumen tersebut dan tidak muncul di dokumen lain, seperti "modern", "mengolah", dan
"prosesor", karena IDF-nya tinggi (DF = 1) sekaligus tetap muncul pada dokumen (TF = 1),
sehingga term tersebut sangat khas/distingtif untuk dokumen ini. Pola yang sama terjadi pada
Dokumen 2 (misalnya "jaringan", "menghubungkan", "perangkat"), Dokumen 3 (misalnya
"kecerdasan", "cabang", "mempelajari"), dan Dokumen 4 (misalnya "informasi",
"dokumen", "temu"). Sebaliknya, term seperti "komputer" yang muncul di banyak dokumen (DF
tinggi) justru memiliki bobot TF-IDF yang relatif rendah meskipun sering muncul, karena IDF-nya
kecil. Hal ini menunjukkan prinsip dasar TF-IDF: term dianggap penting bagi suatu dokumen bukan
hanya karena sering muncul (TF tinggi), tetapi juga karena jarang muncul di dokumen lain (IDF
tinggi), sehingga term tersebut lebih efektif digunakan untuk membedakan dan mengidentifikasi
suatu dokumen dalam sistem pencarian.